# Data Integrity Solver Rule Family

**Status:** Architecture approved; written specification awaiting review  
**Date:** 2026-08-21  
**Design epic:** `bd-1usc`  
**Plan ID:** `f1808cc7-0506-44cf-84d7-235541b689c3`

This notebook is the authoritative design for adding the manifest-backed `data_integrity` family to `spur-solver` with eight seed rules: `unique`, `foreign_key`, `cardinality`, `value_range`, `conditional_required`, `aggregate_balance`, `mutually_consistent`, and `temporal_consistency`.

Formal cells are pinned to the live `relational_lia@1` registry profile: Z3 over quantifier-free Boolean, mathematical integer, and finite-enum constraints. Collection-shaped rules are expanded over bounded facts before solving; no arrays, strings, quantifiers, raw SMT, or unbounded inference enter the solver request.

## Decision, scope, and integration architecture

Implement one family, one profile (`finite_relational_snapshot`), and one native compiler over a normalized bounded relational snapshot. This gives all eight rules one strict schema, one unknown declaration model, deterministic variable naming, and stable per-rule attribution.

The additive integration follows the existing hybrid catalog:

- Add `crates/spur-solver/src/rules/families/data_integrity.rs` and a `data_integrity/` directory containing `family.yaml`, compiler/model helpers, and one YAML manifest per rule.
- Register the compiler in `families::compilers()`.
- Extend `NativeHandlerV1` with eight closed, family-owned handler variants whose parameter ABIs are exact and manifest-validated.
- Keep rule definitions in family facts maps (`unique_constraints`, `foreign_keys`, `cardinality_constraints`, `value_ranges`, `conditional_requirements`, `aggregate_balances`, `consistency_relations`, and `temporal_constraints`). Each binding names exactly one definition ID as its subject.
- Lower only to existing typed `SolveConstraintsRequest` expressions and preserve the executor's solver status, UNSAT-core names, timeout/resource outcomes, caller ordering, and verification attribution.

Out of scope: database mutation, triggers, SQL transaction semantics, unbounded relation inference, decimal rounding/tolerance, runtime raw-SMT manifests, and automatic schema coercion.

## Typed finite relational snapshot

Facts declare finite relations, strict field schemas, bounded rows, cells, and optional unknown declarations.

Let `a_r ∈ Bool` mean row `r` is active, `p_(r,f) ∈ Bool` mean field `f` is present, and `v_(r,f)` be its typed value. Field domains are:

- bounded mathematical integer;
- finite enum labels declared in caller order;
- Boolean.

References resolve by ID before variable allocation. Duplicate relation, row, field, rule-definition, enum-label, or unknown IDs are compilation errors. Every fixed value must match its field domain exactly.

### Null and unknown policy

The seed uses SQL-like null behavior without importing SQL's three-valued logic:

- `present: false, value: null` is a complete absent value.
- A synthesis hole exists only when backed by an explicit bounded unknown declaration.
- Unknown kinds are `row_active`, `cell_present`, and `cell_value`; a value unknown inherits its field domain.
- Verification rejects every unknown declaration and every incomplete cell fact.
- `unique` uses NULLS DISTINCT: rows with an incomplete key do not conflict.
- `foreign_key` uses MATCH SIMPLE: an incomplete child key does not require a parent match.
- `value_range` ignores absent values; `conditional_required` owns requiredness.

This separation prevents an omitted fact from silently becoming a solver variable.

In [ ]:
flowchart TD
    SPEC["`@spec DATA-INTEGRITY-CELL-MODES
@type Status = enum[absent, fixed, unknown, invalid]
@input present: Bool
@input literal_value: Bool
@input unknown_declared: Bool
@output status: Status
@requires PRE: true`"]

    ABSENT["`@branch ABSENT
@when not present and not literal_value and not unknown_declared
@ensures ABSENT_STATUS: status = absent`"]

    FIXED["`@branch FIXED
@when present and literal_value and not unknown_declared
@ensures FIXED_STATUS: status = fixed`"]

    UNKNOWN["`@branch UNKNOWN
@when unknown_declared and not literal_value
@ensures UNKNOWN_STATUS: status = unknown`"]

    INVALID["`@branch INVALID
@when not ((not present and not literal_value and not unknown_declared) or (present and literal_value and not unknown_declared) or (unknown_declared and not literal_value))
@ensures INVALID_STATUS: status = invalid`"]

    CHECK["`@verify CELL_MODE_DETERMINISTIC: prove determinism
@verify CELL_MODE_COVERAGE: prove partition_coverage
@verify CELL_MODE_EXCLUSIVE: prove partition_exclusive
@verify CELL_MODE_STATUSES: witness each status`"]

    SPEC --> ABSENT --> CHECK
    SPEC --> FIXED --> CHECK
    SPEC --> UNKNOWN --> CHECK
    SPEC --> INVALID --> CHECK

## Mathematical rule contracts

For a key field set `K`, write `complete_K(r) = ∧_(f∈K) p_(r,f)`.

1. **`data_integrity.unique` — NULLS DISTINCT**

   For every ordered active row pair `r < s`:
   `a_r ∧ a_s ∧ complete_K(r) ∧ complete_K(s) ⇒ ∨_(f∈K) (v_(r,f) ≠ v_(s,f))`.

2. **`data_integrity.foreign_key` — MATCH SIMPLE**

   For each child row `c` with child key `C` and corresponding parent key `P`:
   `a_c ∧ complete_C(c) ⇒ ∨_(p∈ParentRows) [a_p ∧ complete_P(p) ∧ ∧_i(v_(c,C_i) = v_(p,P_i))]`.

3. **`data_integrity.cardinality`**

   For a relation or declared row subset `R`:
   `minimum ≤ Σ_(r∈R) ite(a_r, 1, 0) ≤ maximum`.
   The compiler expands the Boolean count into typed linear terms.

4. **`data_integrity.value_range`**

   For an integer field and inclusive bounds:
   `a_r ∧ p_(r,f) ⇒ minimum ≤ v_(r,f) ≤ maximum`.

5. **`data_integrity.conditional_required`**

   For a typed equality predicate `v_(r,t) = expected`:
   `a_r ∧ p_(r,t) ∧ (v_(r,t) = expected) ⇒ p_(r,required)`.

6. **`data_integrity.aggregate_balance`**

   For declared integer terms `T` with coefficients `c_t`:
   `[∧_(t∈T) (a_(row(t)) ⇒ p_t)] ∧ Σ_(t∈T) c_t · v_t = target`.
   Arithmetic is exact mathematical integer arithmetic; no tolerance or rounding is inferred.

7. **`data_integrity.mutually_consistent`**

   For fields `F` and finite allowed tuple relation `A`:
   `a_r ⇒ complete_F(r) ∧ ∨_(tuple∈A) ∧_i(v_(r,F_i) = tuple_i)`.

8. **`data_integrity.temporal_consistency`**

   For every active interval row, `p_start ∧ p_end ∧ start < end`. For each declared predecessor edge `before → after`:
   `a_after ⇒ a_before ∧ end_before ≤ start_after`.

All implications lower as `not antecedent or consequent`; tuple distinctness and joins are finitely expanded, not encoded with quantifiers.

In [ ]:
flowchart TD
    SPEC["`@spec DATA-INTEGRITY-FINITE-INSTANCE
@type Plan = enum[free, pro]
@type Region = enum[us, eu]
@input row_count: Int
@input unique_a_active: Bool
@input unique_b_active: Bool
@input unique_a_present: Bool
@input unique_b_present: Bool
@input unique_a_key: Int
@input unique_b_key: Int
@input child_active: Bool
@input child_fk_present: Bool
@input child_fk: Int
@input parent_active: Bool
@input parent_key_present: Bool
@input parent_key: Int
@input measured_present: Bool
@input measured_value: Int
@input trigger_matches: Bool
@input required_present: Bool
@input balance_left: Int
@input balance_delta: Int
@input balance_right: Int
@input plan: Plan
@input region: Region
@input predecessor_active: Bool
@input predecessor_end: Int
@input successor_active: Bool
@input successor_start: Int
@input successor_end: Int
@requires CARDINALITY: row_count >= 1 and row_count <= 4
@requires UNIQUE: not (unique_a_active and unique_b_active and unique_a_present and unique_b_present) or unique_a_key != unique_b_key
@requires FOREIGN_KEY: not (child_active and child_fk_present) or (parent_active and parent_key_present and child_fk = parent_key)
@requires VALUE_RANGE: not (child_active and measured_present) or (measured_value >= 0 and measured_value <= 100)
@requires CONDITIONAL_REQUIRED: not (child_active and trigger_matches) or required_present
@requires AGGREGATE_BALANCE: balance_left + balance_delta = balance_right
@requires MUTUALLY_CONSISTENT: (plan = free and region = us) or (plan = pro and region = us) or (plan = pro and region = eu)
@requires TEMPORAL_INTERVAL: not successor_active or successor_start < successor_end
@requires TEMPORAL_PREDECESSOR: not successor_active or (predecessor_active and predecessor_end <= successor_start)
@requires KEY_BOUNDS: unique_a_key >= 0 and unique_a_key <= 1000 and unique_b_key >= 0 and unique_b_key <= 1000 and child_fk >= 0 and child_fk <= 1000 and parent_key >= 0 and parent_key <= 1000
@requires BALANCE_BOUNDS: balance_left >= -1000 and balance_left <= 1000 and balance_delta >= -1000 and balance_delta <= 1000 and balance_right >= -2000 and balance_right <= 2000
@requires TIME_BOUNDS: predecessor_end >= 0 and predecessor_end <= 1000 and successor_start >= 0 and successor_start <= 1000 and successor_end >= 0 and successor_end <= 1001`"]

    CHECK["`@verify INTEGRITY_NONVACUOUS: witness non_vacuity
@verify INTEGRITY_CONSISTENT: witness consistency`"]

    SPEC --> CHECK

## Compiler validation, model budget, and projection

Compilation is fail-closed and deterministic:

1. Deserialize strict facts and reject unknown JSON fields.
2. Validate family profile, rule ownership, exact handler ABI, and one binding subject per rule.
3. Index and validate all relation/field/row/definition IDs, field domains, tuple arity, key arity/type compatibility, interval fields, aggregate terms, and bound ordering.
4. Reject unknowns in verification; in synthesis allocate variables only for explicit unknown declarations.
5. Compute a checked pre-lowering AST estimate and reject overflow or budget excess before constructing solver expressions.
6. Allocate typed variables in caller-declared order, compile one named `CompiledRule` per binding, validate the resulting generic request, solve, then project values and violations back in caller order.

The initial aggregate AST budget is `MAX_CONSTRAINTS × MAX_VARIABLES = 16,384` nodes, using checked arithmetic. Estimates include:

- unique: `O(rows² × key_fields)`;
- foreign key: `O(child_rows × parent_rows × key_fields)`;
- cardinality and ranges: `O(rows)`;
- aggregate balance: `O(terms)`;
- mutual consistency: `O(rows × allowed_tuples × fields)`;
- temporal consistency: `O(rows + predecessor_edges)`.

Every generated constraint name carries the rule binding and definition subject, so logical conflicts remain solver `unsat` outcomes with attributable cores. Invalid schemas, dangling references, incompatible domains, malformed parameters, and budget failures are compilation errors.

In [ ]:
flowchart TD
    SPEC["`@spec DATA-INTEGRITY-RELEASE-GATE
@type Status = enum[accepted, rejected]
@input unique_ready: Bool
@input foreign_key_ready: Bool
@input cardinality_ready: Bool
@input value_range_ready: Bool
@input conditional_required_ready: Bool
@input aggregate_balance_ready: Bool
@input mutually_consistent_ready: Bool
@input temporal_consistency_ready: Bool
@input strict_schema_ready: Bool
@input checked_budget_ready: Bool
@input dual_evaluation_ready: Bool
@output status: Status
@requires PRE: true`"]

    ACCEPTED["`@branch ACCEPTED
@when unique_ready and foreign_key_ready and cardinality_ready and value_range_ready and conditional_required_ready and aggregate_balance_ready and mutually_consistent_ready and temporal_consistency_ready and strict_schema_ready and checked_budget_ready and dual_evaluation_ready
@ensures ACCEPTED_STATUS: status = accepted`"]

    REJECTED["`@branch REJECTED
@when not (unique_ready and foreign_key_ready and cardinality_ready and value_range_ready and conditional_required_ready and aggregate_balance_ready and mutually_consistent_ready and temporal_consistency_ready and strict_schema_ready and checked_budget_ready and dual_evaluation_ready)
@ensures REJECTED_STATUS: status = rejected`"]

    CHECK["`@verify RELEASE_DETERMINISTIC: prove determinism
@verify RELEASE_COVERAGE: prove partition_coverage
@verify RELEASE_EXCLUSIVE: prove partition_exclusive
@verify RELEASE_STATUSES: witness each status`"]

    SPEC --> ACCEPTED --> CHECK
    SPEC --> REJECTED --> CHECK

## Verification strategy, research basis, and risks

Every seed rule must receive double evaluation through the family API:

- a valid complete verification vector;
- an invalid complete verification vector asserting exact rule/subject attribution;
- a bounded synthesis vector for each supported unknown kind;
- a malformed-fact or dangling-reference rejection vector;
- manifest ABI and catalog-discovery coverage.

Family-level scenarios compose all eight rules into one satisfiable snapshot and one logically conflicting snapshot whose UNSAT core names the responsible bindings. Budget boundary tests cover the largest accepted shape, first rejected shape, and checked-arithmetic overflow.

The semantics align with primary standards and established formulations:

- PostgreSQL documents CHECK, UNIQUE, foreign keys, composite keys, NULLS DISTINCT/NOT DISTINCT, MATCH SIMPLE/FULL, and temporal `WITHOUT OVERLAPS` / `PERIOD` constraints: [DDL Constraints](https://www.postgresql.org/docs/current/ddl-constraints.html) and [CREATE TABLE](https://www.postgresql.org/docs/current/sql-createtable.html).
- W3C SHACL defines `minCount`, `maxCount`, and inclusive/exclusive value bounds for declarative validation: [SHACL 1.2 Core](https://www.w3.org/TR/shacl12-core/).
- XBRL calculation requirements define summation consistency from weighted contributing facts to a total: [Calculation Requirements 1.1](https://www.xbrl.org/REQ/calculation-requirements-1.1/REQ-2022-05-25/calculation-requirements-1.1-2022-05-25.html).
- Allen's interval algebra supplies the established before/after interval relation vocabulary: [Maintaining Knowledge about Temporal Intervals](https://cse.unl.edu/~choueiry/Documents/Allen-CACM1983.pdf).

Primary risks and mitigations:

- **Relational blow-up:** bounded facts plus checked pre-lowering AST estimation.
- **Null ambiguity:** explicit `present` and explicit unknown declarations with mode-specific validation.
- **Type drift:** strict schemas, composite-key arity/type checks, and no coercion.
- **Misleading aggregates:** exact integer semantics only; decimal tolerance is out of scope.
- **Attribution drift:** stable deterministic constraint names and caller-order projection.
- **Stale catalog process:** verify manifests with a freshly built `spur-solver` binary; a long-lived MCP server must be reloaded before live discovery checks.

## Formal proof evidence

All native formal cells were preflighted against `relational_lia@1` and executed through the notebook solver. Every mandatory facet matched; all evidence is source-fresh.

| Formal spec | Cell ID | Obligations | Source hash | IR hash | Report hash |
|---|---|---:|---|---|---|
| `DATA-INTEGRITY-CELL-MODES` | `18d2a954-a605-4565-8da6-ffb8321ba104` | 7/7 | `7c21fd07b46793dbd96513353d79f7b2f8e53ed434f52293760bdd294474b529` | `1801f1735ff071b80cc2966382ee2ece81204f4b41d79505d2821ff002026c3a` | `e9792d25a95e69375274fd54001706b5ea1611d6bc37d80be61db9b5879ed423` |
| `DATA-INTEGRITY-FINITE-INSTANCE` | `18d2a954-a605-4565-8da6-ffb8321ba106` | 2/2 | `5ffa4224328963f7d86f0c7f2b58f185f3094526a2d42d7daf4499bc57a274b9` | `37074942ee227d15f1d4e6ad6679766c12fcabae0ffd842b3d5749675c16bb32` | `c28ef4a2cd8d10bd2e987e79b8c468c699a791f4a04216b577c6ecca8e30df6a` |
| `DATA-INTEGRITY-RELEASE-GATE` | `18d2a954-a605-4565-8da6-ffb8321ba108` | 5/5 | `184f6ea651333b3420d5b2acf69da614bfd19740e04cc8a2fb7cbd02a840d199` | `66da8c3a698ec6b89c5fd338300b2db306189f188ebb73f9e366de60f3932770` | `adf78d3bbb4e94980b159b76b343f80362d11f2f9cd00a40bb6501c17c46b3f8` |

These proofs validate the authored finite mathematical relations and release partition. They do not substitute for implementation tests; production acceptance additionally requires manifest ABI tests, per-rule double evaluation, synthesis/verification mode tests, budget boundaries, fresh catalog discovery, focused `spur-solver` tests, formatting, and independent review.